In [1]:
from Bio.Blast import NCBIWWW, NCBIXML
import subprocess
import pandas as pd
from Bio.Seq import Seq
from Bio.Blast import NCBIWWW, NCBIXML


### Ejercicio 1

In [ ]:
secuencia = "GACGGCAGCACAGAGAAGCTTGCTTCTCGGGTGGCGAGTGGCGAACGGGTGAGTAA"
print("Longitud:", len(secuencia), "bases")

resultado = NCBIWWW.qblast(
    "blastn",
    "16S_ribosomal_RNA",
    secuencia,
    hitlist_size=10,
    megablast=False,
)

registro = NCBIXML.read(resultado)

print("\n--- ONLINE: 10 mejores coincidencias ---")
for aln in registro.alignments:
    hsp = aln.hsps[0]
    identidad = 100 * hsp.identities / hsp.align_length
    print(f"{aln.title[:90]}")
    print(f"   E-value: {hsp.expect:.2e} | Identidad: {identidad:.1f}% "
          f"({hsp.identities}/{hsp.align_length}) | Score: {hsp.score}")

In [ ]:
columnas = "stitle pident length mismatch gapopen evalue bitscore"
r = subprocess.run([
    "blastn",
    "-task", "blastn",
    "-query", "paciente_lcr.fasta",
    "-db", "C:/blastdb/16S_ribosomal_RNA",
    "-outfmt", f"6 {columnas}",
    "-max_target_seqs", "10",
    "-evalue", "1e-3",
    "-out", "resultado_local.tsv",
], capture_output=True, text=True)

print("Código de salida:", r.returncode)
print(r.stderr)

df_local = pd.read_csv("resultado_local.tsv", sep="\t", names=columnas.split())
df_local = df_local.sort_values("bitscore", ascending=False)
df_local

### Interpretación (BLAST local)
- La secuencia 16S del paciente coincide con *Neisseria meningitidis* con **100 % de identidad** (56/56 bases, E-value 2·10⁻²²). Todas las demás coincidencias son del mismo género *Neisseria* con 98,2 % (1 diferencia).
- *N. meningitidis* es un patógeno típico de meningitis bacteriana, y los síntomas (rigidez de nuca, fiebre alta, petequias) son compatibles con meningococemia. **Estamos ante una emergencia vital.**
- Limitación: el fragmento es corto (56 pb) y el 16S es muy parecido entre especies de *Neisseria*. Solo 1 base separa al meningococo de especies como *N. gonorrhoeae* o *N. cinerea* (comensal), así que la identificación por 16S es muy fuerte pero no definitiva. Convendría confirmarla con cultivo o PCR específica de otro gen.

### Ejercicio 2

In [ ]:
adn_geiser = "CTCTCCTGGGACCTGGCCAAGGTGCGCACCGACCTGCCCCTGGAGGTGGACTTCGCCAAAAGGCGGGAGC"
adn = Seq(adn_geiser.strip())
print("Longitud:", len(adn), "pb")

print("\n--- Los 6 marcos de lectura ---")
for nombre_hebra, secuencia in (("+", adn), ("-", adn.reverse_complement())):
    for marco in range(3):
        sub = secuencia[marco:]
        sub = sub[: len(sub) // 3 * 3]
        print(f"Hebra {nombre_hebra}, marco {marco + 1}: {sub.translate()}")

resultado = NCBIWWW.qblast("blastx", "swissprot", str(adn), hitlist_size=5)
with open("blastx_geiser.xml", "w") as f:
    f.write(resultado.read())

registro = NCBIXML.read(open("blastx_geiser.xml"))
print("\n--- Mejores coincidencias (blastx) ---")
for aln in registro.alignments:
    hsp = aln.hsps[0]
    print(aln.title[:100])
    print(f"   E-value: {hsp.expect:.2e} | Identidad: "
          f"{100 * hsp.identities / hsp.align_length:.1f}% | Marco: {hsp.frame}")

print("\nProteína descubierta:", registro.alignments[0].title)

Longitud: 70 pb

--- Los 6 marcos de lectura ---
Hebra +, marco 1: LSWDLAKVRTDLPLEVDFAKRRE
Hebra +, marco 2: SPGTWPRCAPTCPWRWTSPKGGS
Hebra +, marco 3: LLGPGQGAHRPAPGGGLRQKAG
Hebra -, marco 1: APAFWRSPPPGAGRCAPWPGPRR
Hebra -, marco 2: LPPFGEVHLQGQVGAHLGQVPGE
Hebra -, marco 3: SRLLAKSTSRGRSVRTLARSQE


### Explicación
- No se conoce ni la hebra ni el marco de lectura, así que hay seis traducciones posibles: tres marcos en la hebra directa y otros tres en la complementaria inversa.
- Se tradujeron los seis con Biopython. Ninguno tiene codones de parada (el fragmento es corto), así que la secuencia por sí sola no basta para decidir cuál es el real.
- Se usó **blastx**, que traduce el ADN en los seis marcos y compara cada traducción con una base de datos de proteínas. La mejor coincidencia indica el marco correcto y la proteína.
- Resultado: [nombre de la proteína, identidad, E-value y marco].
- Relación con el contexto: [por qué esa proteína es útil en un detergente / es resistente al calor].

### Ejercicio 3

### Ejercicio 4